# 🎓 Chapter 4: Going Deeper

## *Log-odds, bans, information theory, and modern connections*

---

> *This chapter is aimed at undergraduate students with some background
> in probability or information theory. High school readers: come back
> after a course in discrete mathematics!*

---

### Why Turing Didn't Use Bayes' Theorem Directly

Turing understood Bayesian inference perfectly well. But at Bletchley Park
in 1940, there were no computers. Probabilities had to be combined by hand,
using tabulated values and mental arithmetic.

Multiplying lots of probabilities is tedious and numerically unstable
(numbers get very small very quickly). Turing had a better idea.


## Part 1: Log-Odds — The Key Insight

Instead of working with probabilities $P(H)$, Turing worked with **odds**:

$$\text{odds}(H) = \frac{P(H)}{1 - P(H)}$$

Odds of 1 mean $P(H) = 0.5$. Odds of 9 mean $P(H) = 0.9$.

Now consider Bayesian updating. If we have odds $\Omega_0$ before seeing
evidence $E$, and the likelihood ratio is $\Lambda = P(E|H) / P(E|\neg H)$, then:

$$\Omega_1 = \Lambda \cdot \Omega_0$$

Taking $\log_{10}$:

$$\log_{10}(\Omega_1) = \log_{10}(\Lambda) + \log_{10}(\Omega_0)$$

**Multiplication becomes addition.** This is the point. Each piece of evidence
*adds* to the log-odds. No more multiplying chains of small probabilities.

Turing defined the unit of evidence as a **ban** (named after Banbury, where
the codebreaking sheets were printed):

$$1 \text{ ban} = \log_{10}(10) = \text{a factor of 10 in the odds}$$

A **deciban** = 0.1 bans. Turing found that about **3 bans (30 decibans)** of
evidence — meaning 1000:1 odds — was a reliable threshold for declaring
a setting correct.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from src.enigma_bayes.bayes.scoring import to_bans, to_decibans, weight_of_evidence, bans_to_posterior_odds
from src.enigma_bayes.bayes.probability import log_odds

# Demonstrate the ban scale
print('The Ban scale:')
print(f'{"Odds":>12}  {"P(H)":>8}  {"Bans":>8}  {"Decibans":>10}')
print('-' * 45)
for odds_val in [0.1, 1, 10, 100, 1000, 10000]:
    p = odds_val / (1 + odds_val)
    b = np.log10(odds_val)
    db = 10 * b
    print(f'{odds_val:>12.1f}  {p:>8.4f}  {b:>8.2f}  {db:>10.1f}')

In [ ]:
# Sequential Bayesian updating in log-odds space
# Suppose we're testing a hypothesis H: 'This Enigma setting is correct'

# Prior: 1 correct setting out of ~105,000 candidates
n_candidates = 105_456
prior_odds   = 1.0 / n_candidates
log_prior    = np.log10(prior_odds)

print(f'Prior odds:     1 / {n_candidates:,} = {prior_odds:.2e}')
print(f'Prior log-odds: {log_prior:.2f} bans')
print()

# Each time a letter-pair from our crib is consistent with the setting,
# we gain roughly 0.1 bans of evidence (Turing's estimate for a single
# letter match in a valid position)
evidence_per_letter = 0.12  # bans

cumulative = log_prior
print(f'{"Letters checked":>18}  {"Cumulative bans":>16}  {"Posterior odds":>16}')
print('-' * 55)
for n_letters in [0, 5, 10, 20, 30, 40]:
    cumulative = log_prior + n_letters * evidence_per_letter
    post_odds  = 10 ** cumulative
    print(f'{n_letters:>18}  {cumulative:>16.2f}  {post_odds:>16.4f}')

In [ ]:
# Visualise the log-odds trajectory toward 3-ban threshold
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

letters = np.arange(0, 61)
log_odds_vals = log_prior + letters * evidence_per_letter
posterior_probs = 10**log_odds_vals / (1 + 10**log_odds_vals)

ax1.plot(letters, log_odds_vals, color='#c8a44a', linewidth=2.5)
ax1.axhline(y=3, color='#4a90e2', linestyle='--', alpha=0.8, label='3-ban threshold (1000:1 odds)')
ax1.axhline(y=0, color='white', linestyle=':', alpha=0.4)
threshold_letter = int(np.ceil((3 - log_prior) / evidence_per_letter))
ax1.axvline(x=threshold_letter, color='#c8a44a', linestyle=':', alpha=0.6)
ax1.set_xlabel('Letters of crib checked', fontsize=11)
ax1.set_ylabel('Log-odds (bans)', fontsize=11)
ax1.set_title("Turing's Bayesian Scoreboard", fontsize=12, fontweight='bold')
ax1.legend()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

ax2.plot(letters, posterior_probs * 100, color='#c8a44a', linewidth=2.5)
ax2.axhline(y=99.9, color='#4a90e2', linestyle='--', alpha=0.8, label='99.9% confidence')
ax2.set_xlabel('Letters of crib checked', fontsize=11)
ax2.set_ylabel('P(correct setting | evidence) %', fontsize=11)
ax2.set_title('Posterior Probability of Correct Setting', fontsize=12, fontweight='bold')
ax2.legend()
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig.suptitle(f'Threshold reached after ~{threshold_letter} letters of consistent evidence',
             fontsize=11, color='#c8a44a')
plt.tight_layout()
plt.show()

## Part 2: Connection to Information Theory

Claude Shannon published his landmark paper on information theory in 1948,
eight years after Turing developed the ban system at Bletchley. The connection
is not coincidental — both were independently discovering the same deep truth.

Shannon defined **entropy** as the average uncertainty in a random variable:

$$H(X) = -\sum_{i} p_i \log_2(p_i) \quad \text{(bits)}$$

A **bit** of information (base-2) is related to Turing's **ban** (base-10) by:

$$1 \text{ ban} = \log_2(10) \approx 3.32 \text{ bits}$$

The **Kullback-Leibler (KL) divergence** measures how much information a
posterior distribution contains compared to a prior:

$$D_{KL}(P \| Q) = \sum_i P(i) \log \frac{P(i)}{Q(i)}$$

Each piece of evidence reduces our uncertainty by $D_{KL}(\text{posterior} \| \text{prior})$
bits — exactly what Turing was measuring with his bans.


In [ ]:
import numpy as np
from scipy.stats import entropy as scipy_entropy

def kl_divergence(posterior: np.ndarray, prior: np.ndarray) -> float:
    """KL divergence from prior to posterior (nats → convert to bits)."""
    # Only sum over non-zero posterior entries
    mask = posterior > 0
    return float(np.sum(posterior[mask] * np.log2(posterior[mask] / prior[mask])))

def shannon_entropy(probs: np.ndarray) -> float:
    """Shannon entropy in bits."""
    return float(scipy_entropy(probs, base=2))

# Trace entropy reduction through Bayesian updating
from src.enigma_bayes.bayes.probability import uniform_prior, bayesian_update

N = 1000  # hypotheses
prior = uniform_prior(N)

# Simulate evidence that concentrates probability on hypothesis 42
rng = np.random.default_rng(0)
print(f'{"Update":>8}  {"Entropy (bits)":>16}  {"KL from prior":>16}  {"P(H_true)":>12}')
print('-' * 57)

p = prior.copy()
for step in range(6):
    # Likelihood: true hypothesis 10x more likely than others
    lh = rng.uniform(0.01, 0.1, N)
    lh[42] = 1.0
    p = bayesian_update(p, lh)
    H     = shannon_entropy(p)
    kl    = kl_divergence(p, prior)
    p_true = p[42]
    print(f'{step+1:>8}  {H:>16.4f}  {kl:>16.4f}  {p_true:>12.6f}')

## Part 3: Modern Connections

The machinery we've built in this course appears throughout modern machine
learning and statistics under different names:

| Bletchley Park concept | Modern name | Where you'll see it |
|---|---|---|
| Crib scoring | Naive Bayes classifier | Spam filters, text classification |
| Prior + likelihood update | Bayesian inference | Medical diagnosis, A/B testing |
| Log-odds (bans) | Logit / log-likelihood ratio | Logistic regression |
| Sequential updating | Online learning | Real-time recommendation systems |
| Index of Coincidence | Chi-squared statistic | Hypothesis testing |
| Enigma constraint (zero likelihood) | Hard constraints in probabilistic models | Physics-informed neural networks |

The Naive Bayes classifier — still used today for spam detection — is
exactly the Bayesian decoder we built in Chapter 3, applied to classifying
documents instead of decrypting ciphertexts.


In [ ]:
# Naive Bayes text classifier — the modern Bletchley decoder
# Classify a message as 'German military' or 'other' based on letter frequencies

from src.enigma_bayes.bayes.probability import uniform_prior, bayesian_update
import numpy as np

# Simplified: German military messages overuse certain letters
# (in practice, letter frequencies differ between languages and message types)
GERMAN_FREQ = {
    'E': 0.165, 'N': 0.100, 'I': 0.076, 'S': 0.073, 'R': 0.070,
    'A': 0.065, 'T': 0.061, 'D': 0.051, 'H': 0.048, 'U': 0.044,
}

ENGLISH_FREQ = {
    'E': 0.127, 'T': 0.091, 'A': 0.082, 'O': 0.075, 'I': 0.070,
    'N': 0.067, 'S': 0.063, 'H': 0.061, 'R': 0.060, 'D': 0.043,
}

def naive_bayes_language(text: str, freq_H0: dict, freq_H1: dict) -> float:
    """Return log P(H0=German | text) - log P(H1=English | text) in bans."""
    text = ''.join(c for c in text.upper() if c.isalpha())
    score = 0.0
    for letter in text:
        p0 = freq_H0.get(letter, 0.01)
        p1 = freq_H1.get(letter, 0.01)
        score += np.log10(p0 / p1)  # each letter adds/subtracts bans
    return score

tests = [
    ('WETTERVORHERSAGE', 'German weather report'),
    ('NOTHINGTOREPORTHERETODAY', 'English "nothing to report"'),
    ('EINEINSEINSNULLDREI', 'German encoded number'),
]

print(f'{"Text":<30} {"Score (bans)":>15}  Classification')
print('-' * 65)
for text, label in tests:
    score = naive_bayes_language(text, GERMAN_FREQ, ENGLISH_FREQ)
    cls = 'German' if score > 0 else 'English'
    print(f'{text:<30} {score:>+15.3f}  → {cls}  ({label})')

## Summary and Further Reading

In this advanced chapter we explored:

1. **Log-odds (bans/decibans)**: Turing's unit of evidence. Addition of bans
   = multiplication of likelihoods. Each ban = 10× change in odds.

2. **Sequential updating**: each crib letter adds ~0.1–0.2 bans. Around 30
   letters of consistent evidence achieves 3-ban confidence (~1000:1 odds).

3. **Information theory**: entropy measures uncertainty; KL divergence measures
   how much information evidence provides; 1 ban ≈ 3.32 bits.

4. **Modern applications**: Naive Bayes, logistic regression, and online
   learning are all descendants of the same machinery.

### Further Reading

- Turing, A.M. (c. 1941). *Treatise on the Enigma* (declassified 1996). HW 25/37, UK National Archives.
- Good, I.J. (1979). *Studies in the History of Probability and Statistics. XXXVII. A.M. Turing's Statistical Work in World War II*. Biometrika, 66(2), 393–396.
- MacKay, D.J.C. (2003). *Information Theory, Inference, and Learning Algorithms*. Cambridge University Press. (Free online.)
- Gelman, A. et al. (2013). *Bayesian Data Analysis* (3rd ed.). Chapman & Hall.
